In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor


In [2]:
filepath = r"C:\Users\felix\ames_housing-ml\data\AmesHousing.csv"
df = pd.read_csv(filepath)


df = df.drop(columns=["Order", "PID"], errors="ignore")
df = df[df['Gr Liv Area'] <= 4000]



In [3]:
df["TotalSF"] = (
    df["Total Bsmt SF"] +
    df["1st Flr SF"] +
    df["2nd Flr SF"]
)

df["HouseAge"] = df["Yr Sold"] - df["Year Built"]
df["RemodAge"] = df["Yr Sold"] - df["Year Remod/Add"]


In [4]:
X = df.drop(columns=["SalePrice"])
y = np.log1p(df["SalePrice"])  


In [5]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()


for col in categorical_cols:
    X[col] = X[col].fillna("Missing")





In [6]:
cat_features = [X.columns.get_loc(col) for col in categorical_cols]


In [7]:
model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.02,
    depth=6,
    l2_leaf_reg=3,
    cat_features = cat_features,
    loss_function="MAE",
    random_seed=42,
    verbose=0
)


In [ ]:
cv = RepeatedKFold(
    n_splits=5,
    n_repeats=7,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

mae_mean = -scores.mean()
mae_std = scores.std()

print("MAE (log scale) mean:", mae_mean)
print("MAE (log scale) std :", mae_std)


In [ ]:
import matplotlib.pyplot as plt


model.fit(X, y)


feature_importance = model.get_feature_importance()
sorted_idx = np.argsort(feature_importance)
plt.figure(figsize=(10, 10))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(X.columns)[sorted_idx])
plt.title('Fitur Paling Berpengaruh')
plt.show()